In [1]:
# Import Library
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import random

c:\SEM 5\LAB AI Semester 5\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
# Load data
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = 'sample_audio.wav'
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

In [3]:
def export_pydub(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be ends with .wav")
    audio.export(file_name, format="wav")
    print("Audio saved as", file_name)
    
def export_librosa(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be ends with .wav")
    sf.write(file_name, np.clip(audio, -1,1), sr)
    print("Audio saved as", file_name)

In [4]:
# Predict Text Function

def predict_text(audio_file):
    waveform, sr = torchaudio.load(audio_file)
    waveform = waveform.mean(dim=0, keepdim=True)
    
    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
    
    with torch.inference_mode():
        emission, _ = model(waveform)
        
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0
            
        def forward(self, emission: torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])
        
    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript
            

In [5]:
# Augmentation Function

def adjvolume(clip):
    while True:
        db = float(input("Input DB adjust volume (-30 to 30): "))
        if -30 <= db <= 30:
            print("must be between -30 to 30")
        else:
            break
    export_pydub(clip + db)
    
def repeat(clip):
    while True:
        repeat = int(input("Input repeat audio (1 to 50): "))
        if 1 <= repeat <= 50:
            print("must be between 1 to 50")
        else:
            break
    export_pydub(clip * repeat)
    
def fade_in(clip):
    while True:
        duration = int(input("Input fadein duration in ms (1s = 1000ms): "))
        if duration > 0:
            print("must be more than 0")
        else:
            break
    export_pydub(clip.fade_in(duration))
    
def fade_out(clip):
    while True:
        duration = int(input("Input fadeout duration in ms (1s = 1000ms): "))
        if duration > 0:
            print("must be more than 0")
        else:
            break
    export_pydub(clip.fade_out(duration))

def reverse_audio(clip):
    export_pydub(clip.reverse())
    
def adjpitch():
    while True:
        pitch_rate = float(input("Input pitch rate (-5 to 5): "))
        if -5 <= pitch_rate <= 5:
            print("must be between -5 to 5")
        else:
            break
    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_librosa(shifted_audio)
    
def adjspeed():
    while True:
        rate = float(input("Input pitch rate (normal = 1) (0.5 to 2): "))
        if 0.5 <= rate <= 2:
            print("must be between 0.5 to 2")
        else:
            break
    speed_audio = librosa.effects.time_stretch(audio, rate=rate)
    export_librosa(speed_audio)

def white_noise():
    while True:
        level = float(input("Input noise level (0.1 to 0.5): "))
        if 0.1 <= level <= 0.5:
            print("must be between 0.1 to 0.5")
        else:
            break
    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * level) 
    
def random_gain():
    gain = np.random.uniform(0.5, 1.5)
    print(f"random gain x {gain:.2f}")
    export_librosa(audio * gain)

In [6]:
def aug_menu():
    while True:
        print("1. Volume")
        print("2. Repeat")
        print("3. Fade In")
        print("4. Fade Out")
        print("5. Reverse")
        print("6. Pitch")
        print("7. Speed")
        print("8. White Noise")
        print("9. Random Gain")
        print("10. Back to Main Menu")
        
        choice = input(">> ")
        if choice == "1":
            adjvolume(clip)
        elif choice == "2":
            repeat(clip)
        elif choice == "3":
            fade_in(clip)
        elif choice == "4":
            fade_out(clip)
        elif choice == "5":
            reverse_audio(clip)
        elif choice == "6":
            adjpitch()
        elif choice == "7":
            adjspeed()
        elif choice == "8":
            white_noise()
        elif choice == "9":
            random_gain()
        elif choice == "10":
            break
        else:
            print("Invalid Input must be (1 to 10)")

In [7]:
def mainmenu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        
        choice = input(">> ")
        if choice == "1":
            print("Audio Augmentation Menu")
            aug_menu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Hasil Transkrip: ")
            print(transcript)
        elif choice == "3":
            break
        else:
            print("Invalid Input")
    print("Main Menu Closed")

mainmenu()

1. Audio Augmentation
2. Predict Text
3. Exit
Audio Augmentation Menu
1. Volume
2. Repeat
3. Fade In
4. Fade Out
5. Reverse
6. Pitch
7. Speed
8. White Noise
9. Random Gain
10. Back to Main Menu
1. Audio Augmentation
2. Predict Text
3. Exit


RuntimeError: Failed to create AudioDecoder for sample_audio.wav: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.13.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading tr